In [ ]:
# Problema: Comparar el procesamiento local por particiones con una agregación global verificable, sin infraestructura distribuida.

from pathlib import Path
import pandas as pd
ROOT = next(path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / "submission").is_dir())
sales = pd.DataFrame([["P1","Alimentos",20],["P2","Alimentos",10],["P3","Oficina",15],["P4","Oficina",5]], columns=["product_id","product_category","sales_amount"])
# La partición hace visible una organización física; no cambia el resultado lógico.
partitions = [sales.iloc[:2], sales.iloc[2:]]
partial = [part.groupby("product_category", as_index=False).sales_amount.sum() for part in partitions]
result = pd.concat(partial).groupby("product_category", as_index=False).sales_amount.sum().sort_values("product_category")
reference = sales.groupby("product_category", as_index=False).sales_amount.sum().sort_values("product_category")
assert result.reset_index(drop=True).equals(reference.reset_index(drop=True))
result.to_parquet(ROOT / "submission/category_sales.parquet", index=False)
pd.DataFrame([["input_rows",len(sales)],["teaching_input_partitions",len(partitions)],["shuffle_partitions",len(partitions)],["output_categories",len(result)],["validation_status","PASS"]], columns=["metric","value"]).to_csv(ROOT / "submission/execution_summary.csv", index=False)